# Exp0.1.4 — Parallel Regularization Repair Ablation

Analysis-only notebook for the 30-epoch paired regularization repair sweep. The equal-budget control is `no_reg`; frozen Exp0.1 100-epoch results are historical context only.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_repo_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "AGENTS.md").exists() and (candidate / "scripts").exists():
            return candidate
    raise FileNotFoundError("Could not locate writingRing repository root")


REPO_ROOT = find_repo_root()
ROOT = REPO_ROOT / "notebooks" / "artifacts" / "experiment_0_1_4_parallel_regularization_ablation" / "parallel_regularization_ablation_v1"

manifest = json.loads((ROOT / "manifest.json").read_text(encoding="utf-8"))
runs = pd.read_csv(ROOT / "runs.csv")
summary = pd.read_csv(ROOT / "summary.csv")
history = pd.read_csv(ROOT / "history_long.csv")
paired = pd.read_csv(ROOT / "paired_condition_effects.csv")
paired_summary = pd.read_csv(ROOT / "paired_condition_summary.csv")
gradient_summary = pd.read_csv(ROOT / "gradient_diagnostics_summary.csv")
frozen = pd.read_csv(ROOT / "frozen_exp01_summary.csv")
comparison = pd.read_csv(ROOT / "comparison_summary.csv")

print(json.dumps(manifest["run_matrix"], indent=2))
print(f"runs={len(runs)} history_rows={len(history)} paired_rows={len(paired)}")


## Five-seed test BA by condition

The main table compares all six 30-epoch conditions on identical architecture/objective/seed pairs.


In [ ]:
ba_table = summary.pivot_table(
    index=["architecture", "objective"],
    columns="condition",
    values="test_ba_mean",
)
condition_order = manifest["run_matrix"]["conditions"]
ba_table = ba_table.reindex(columns=[c for c in condition_order if c in ba_table.columns])
display(ba_table.round(4))


## Paired delta relative to 30-epoch no-reg control

Positive `delta_test_ba_vs_no_reg` means the regularization condition improved over the equal-budget baseline.


In [ ]:
delta_table = paired_summary.pivot_table(
    index=["architecture", "objective"],
    columns="condition",
    values="delta_test_ba_vs_no_reg_mean",
)
display(delta_table.round(4))

best = (
    paired_summary.sort_values("delta_test_ba_vs_no_reg_mean", ascending=False)
    [["condition", "architecture", "objective", "delta_test_ba_vs_no_reg_mean", "delta_test_ba_vs_no_reg_std"]]
)
display(best.head(18).round(4))


## Firing preservation and dead-neuron behavior

These paired metrics test whether a condition improves BA by preserving useful firing rather than simply reducing membrane magnitude.


In [ ]:
firing_cols = [
    "condition",
    "architecture",
    "objective",
    "delta_last_hidden_events_per_neuron_second_vs_no_reg_mean",
    "delta_last_hidden_dead_neuron_fraction_vs_no_reg_mean",
    "delta_test_ba_vs_no_reg_mean",
]
display(
    paired_summary[firing_cols]
    .sort_values(["architecture", "objective", "condition"])
    .round(4)
)


## 30-epoch no-reg versus frozen 100-epoch Exp0.1 context

This is not a paired equal-budget comparison. It only indicates how much performance may still be left from extending training beyond 30 epochs.


In [ ]:
no_reg = summary[summary["condition"] == "no_reg"][
    ["architecture", "objective", "test_ba_mean", "test_ba_std"]
].rename(columns={"test_ba_mean":"no_reg_30ep_mean", "test_ba_std":"no_reg_30ep_std"})
context = frozen.rename(columns={"mean":"exp01_100ep_mean", "std":"exp01_100ep_std"})
context_compare = no_reg.merge(context, on=["architecture", "objective"], how="left")
context_compare["delta_100ep_minus_30ep"] = context_compare["exp01_100ep_mean"] - context_compare["no_reg_30ep_mean"]
display(context_compare.round(4))


## Gradient competition diagnostics

Gradient ratios are measured on the first training batch only at epochs 1, 2, 5, 10, 20, and 30.


In [ ]:
grad_cols = [
    "condition",
    "architecture",
    "objective",
    "epoch",
    "first_batch_p2_to_task_grad_ratio_mean",
    "first_batch_a1_to_task_grad_ratio_mean",
    "first_batch_reg_to_task_grad_ratio_mean",
    "train_reg_to_task_ratio_mean",
    "val_native_ba_mean",
]
display(
    gradient_summary[grad_cols]
    .sort_values(["architecture", "objective", "condition", "epoch"])
    .round(4)
)


## Validation BA trajectories

Each line is the five-seed mean for one condition. One figure is generated per architecture/objective pair.


In [ ]:
curve = (
    history.groupby(["condition", "architecture", "objective", "epoch"], as_index=False)["val_native_ba"]
    .agg(["mean", "std"])
    .reset_index()
)

for architecture in manifest["run_matrix"]["architectures"]:
    for objective in manifest["run_matrix"]["objectives"]:
        fig, ax = plt.subplots(figsize=(8, 4.5))
        subset = curve[(curve["architecture"] == architecture) & (curve["objective"] == objective)]
        for condition in condition_order:
            part = subset[subset["condition"] == condition]
            if part.empty:
                continue
            ax.plot(part["epoch"], part["mean"], marker="o", markevery=[0, 4, 9, 19, 29], label=condition)
        ax.set_title(f"{architecture} | {objective}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Validation balanced accuracy")
        ax.set_ylim(0.0, 1.0)
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8, ncol=2)
        plt.show()


## Gradient-ratio trajectories at diagnostic epochs

A ratio above 1 means the combined weighted regularizer gradient is larger than the task gradient on that diagnostic batch.


In [ ]:
grad_curve = gradient_summary.copy()
for architecture in manifest["run_matrix"]["architectures"]:
    for objective in manifest["run_matrix"]["objectives"]:
        fig, ax = plt.subplots(figsize=(8, 4.5))
        subset = grad_curve[(grad_curve["architecture"] == architecture) & (grad_curve["objective"] == objective)]
        for condition in condition_order:
            part = subset[subset["condition"] == condition]
            if part.empty:
                continue
            ax.plot(
                part["epoch"],
                part["first_batch_reg_to_task_grad_ratio_mean"],
                marker="o",
                label=condition,
            )
        ax.axhline(1.0, linestyle="--", linewidth=1)
        ax.set_yscale("symlog", linthresh=1e-3)
        ax.set_title(f"Gradient competition | {architecture} | {objective}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("||grad regularizer|| / ||grad task||")
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8, ncol=2)
        plt.show()


## Condition ranking

This final table ranks conditions independently inside each architecture/objective pair by mean test BA, while retaining firing and dead-neuron diagnostics.


In [ ]:
ranking = summary[[
    "condition",
    "architecture",
    "objective",
    "test_ba_mean",
    "test_ba_std",
    "test_last_hidden_events_per_neuron_second_mean",
    "test_last_hidden_dead_neuron_fraction_mean",
]].copy()
ranking["rank_within_arch_objective"] = ranking.groupby(["architecture", "objective"])["test_ba_mean"].rank(method="min", ascending=False)
display(
    ranking.sort_values(["architecture", "objective", "rank_within_arch_objective"])
    .round(4)
)
